clerq works in a consistent set of *natural* units, and the `clerq.macro` module converts the numbers you have in the lab (intensities, powers, temperatures, densities) into them.

- Frequencies are **cyclic** ($\Omega/2\pi$, $\Gamma/2\pi$, detunings), in MHz, and times are in µs.
- Lengths are in cm and number densities in cm$^{-3}$.
- The `macro` functions take wavelengths in nm, intensities in mW/cm$^2$, powers in mW and beam waists in cm.

Every function returns a plain number ready to put in a system dictionary or a `PropagationSolver`.

In [1]:
import numpy as np

from clerq import macro

# Rb87 D2 line
decay_rate = 6.065  # Gamma / 2π, MHz
wavelength = 780.24  # nm

## Intensity to Rabi frequency

For a closed cycling transition the saturation intensity is $I_\mathrm{sat} = \pi h c \Gamma / 3\lambda^3$ and the Rabi frequency is $\Omega = \Gamma\sqrt{I / 2 I_\mathrm{sat}}$. Use `factor` to scale the dipole moment squared for a non-cycling transition, for example a squared Clebsch-Gordan coefficient.

In [2]:
i_sat = macro.saturation_intensity(decay_rate, wavelength)  # mW/cm^2
peak_intensity = macro.gaussian_beam_peak_intensity(power=1.0, waist=0.1)  # 1 mW, 1 mm waist
rabi_freq = macro.intensity_to_rabi_freq(peak_intensity, decay_rate, wavelength)
print(f"I_sat = {i_sat:.3f} mW/cm^2, peak = {peak_intensity:.1f} mW/cm^2, Omega/2π = {rabi_freq:.2f} MHz")

I_sat = 1.669 mW/cm^2, peak = 63.7 mW/cm^2, Omega/2π = 26.49 MHz


`macro.rabi_freq_to_intensity` is the inverse. The result is the `rabi_freq` (or `ampl` of a time function) of a clerq `Field`.

## Number density and interaction strength

The propagation equation is $\partial_z\Omega = 2 i N g \rho_{01}$, where $g/2\pi$ is `macro.interaction_strength`, per atom, and $N$ is the number density. Multiply by the density to get the `interaction_strengths` entry. A weak resonant field is absorbed with optical depth $N\sigma_0 L$, with $\sigma_0 = 3\lambda^2/2\pi$.

In [3]:
n = 1e11  # cm^-3
length = 0.5  # cm
g = macro.interaction_strength(decay_rate, wavelength)
interaction_strengths = [n * g]  # MHz / cm
od = macro.resonant_optical_depth(n, length, wavelength)
print(f"interaction_strengths = {interaction_strengths[0]:.3f} MHz/cm, OD = {od:.1f}")

interaction_strengths = 440.726 MHz/cm, OD = 145.3


## Doppler broadening

`velocity_classes` takes a `thermal_width`, the $w$ in weights $e^{-(\delta/w)^2}$. That is the most probable speed over the wavelength, which `macro.thermal_width` gives from the temperature and mass in amu. The FWHM is $2\sqrt{\ln 2}\,w$.

In [4]:
w = macro.thermal_width(temperature=300.0, mass=86.909, wavelength=wavelength)
print(f"thermal_width = {w:.0f} MHz, FWHM = {2 * np.sqrt(np.log(2)) * w:.0f} MHz")

thermal_width = 307 MHz, FWHM = 511 MHz


## Pulse durations and areas

Pulse shapes take their units from time. `t_funcs.sech` and `t_funcs.gaussian` accept `fwhm` in µs and either an amplitude `ampl` in MHz or an area `n_pi`, so convert a duration in the lab (for example 20 ns = 0.02 µs) by dividing by 1000, and use the Rabi frequency above for `ampl`.